In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("synthetic_transactions.csv")
df.head()

In [ ]:
#code cleaning
df = df.dropna()

df = df[df["Amount"] > 0]

df = df[df["Account"] != df["Counterparty"]]

print("Total rows:", len(df))
print("Unique accounts:", df["Account"].nunique())

In [ ]:
#timestamp processing
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

df["week"] = df["Timestamp"].dt.isocalendar().week

df.head()

In [ ]:
#weekly aggregation
inflow = df[df["Type"]=="IN"].groupby(["Account","week"])["Amount"].sum()

outflow = df[df["Type"]=="OUT"].groupby(["Account","week"])["Amount"].sum()

txn_count = df.groupby(["Account","week"]).size()

weekly = pd.concat([inflow,outflow,txn_count],axis=1)

weekly.columns = ["inflow","outflow","txn_count"]

weekly = weekly.fillna(0).reset_index()

weekly.head()

In [ ]:
all_accounts = weekly["Account"].unique()
all_weeks = range(weekly["week"].min(), weekly["week"].max()+1)

index = pd.MultiIndex.from_product(
    [all_accounts, all_weeks],
    names=["Account","week"]
)

weekly = weekly.set_index(["Account","week"]).reindex(index).fillna(0).reset_index()

weekly.head()

In [ ]:
weekly["net_flow"] = weekly["inflow"] - weekly["outflow"]

weekly["velocity"] = weekly["txn_count"]

weekly["outflow_ratio"] = weekly["outflow"] / (weekly["inflow"] + 1)

weekly.head()

In [ ]:
stats = weekly.groupby("Account").agg({

"inflow":["mean","std"],
"outflow":["mean","std"],
"velocity":["mean","std"]

})

stats.columns = ["_".join(col) for col in stats.columns]

stats = stats.reset_index()

stats.head()    

In [ ]:
weekly = weekly.merge(stats,on="Account")

weekly.head()

In [ ]:
def assign_state(row):

    if row["txn_count"] == 0:
        return "S5"

    if row["inflow"] > row["inflow_mean"] + 2*row["inflow_std"]:
        return "S2"

    if row["outflow"] > row["outflow_mean"] + 2*row["outflow_std"]:
        return "S3"

    if row["velocity"] > row["velocity_mean"] + 2*row["velocity_std"]:
        return "S4"

    return "S1"



weekly["state"] = weekly.apply(assign_state,axis=1)

weekly.head()

weekly["state"].value_counts()

In [ ]:
sequences = {}

for acc, group in weekly.groupby("Account"):

    group = group.sort_values("week")

    sequences[acc] = list(group["state"])

list(sequences.items())[:5]

In [ ]:
states = ["S1","S2","S3","S4","S5"]

state_index = {s:i for i,s in enumerate(states)}

In [ ]:
global_counts = np.zeros((5,5))

for seq in sequences.values():

    for i in range(len(seq)-1):

        s1 = state_index[seq[i]]
        s2 = state_index[seq[i+1]]

        global_counts[s1][s2] += 1

global_counts


global_tpm = global_counts.copy()

for i in range(5):

    if global_tpm[i].sum() > 0:
        global_tpm[i] /= global_tpm[i].sum()

global_tpm

In [ ]:
def sequence_log_likelihood(sequence, tpm):

    log_prob = 0
    transitions = 0

    for i in range(len(sequence)-1):

        s1 = state_index[sequence[i]]
        s2 = state_index[sequence[i+1]]

        prob = tpm[s1][s2]

        if prob > 0:
            log_prob += np.log(prob)
            transitions += 1

    if transitions == 0:
        return None

    return log_prob / transitions

In [ ]:
results = []

for acc, seq in sequences.items():

    if len(seq) < 3:
        continue

    score = sequence_log_likelihood(seq, global_tpm)

    if score is not None:

        results.append({

        "Account": acc,
        "LogLikelihood": score

        })

results_df = pd.DataFrame(results)

results_df.head()

In [ ]:
threshold = np.percentile(results_df["LogLikelihood"],5)

threshold

In [ ]:
results_df["Label"] = results_df["LogLikelihood"].apply(

lambda x: "Mule" if x <= threshold else "Normal"

)

results_df.head()

In [ ]:
mules = results_df[results_df["Label"]=="Mule"]

mules

In [ ]:
results_df.to_csv("results_markov.csv",index=False)

mules.to_csv("detected_mules.csv",index=False)

In [ ]:
plt.hist(results_df["LogLikelihood"],bins=40)

plt.axvline(threshold,color="red",linestyle="--")

plt.title("Account Behavior Likelihood Distribution")

plt.show()